# Ask-SLM CLI Runner (No UI)

This notebook runs Ask-SLM CLI capabilities using a selected workspace only.

Usage flow:
1. Set `WORKSPACE` in Cell 2.
2. Run one or more analysis cells below.
3. Outputs are written into `Workspaces/<WORKSPACE>/results/` by the CLI defaults.

In [ ]:
from pathlib import Path
import shlex
import subprocess
import sys

# Configure this once per run.
WORKSPACE = "ABX"
WORKSPACE_ROOT = Path("../Workspaces")


def run_cli(args: list[str], check: bool = True) -> subprocess.CompletedProcess:
    """Run askslm.py with pretty command logging."""
    cmd = [sys.executable, "askslm.py"] + args
    print("$", " ".join(shlex.quote(part) for part in cmd))
    result = subprocess.run(cmd, text=True)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {' '.join(cmd)}")
    return result


workspace_dir = WORKSPACE_ROOT / WORKSPACE
workspace_inputs = workspace_dir / "inputs"
workspace_configs = workspace_dir / "mapping_configs"
workspace_results = workspace_dir / "results"

print(f"Workspace: {workspace_dir}")
print(f"Inputs:    {workspace_inputs}")
print(f"Configs:   {workspace_configs}")
print(f"Results:   {workspace_results}")

if not workspace_dir.exists():
    raise FileNotFoundError(f"Workspace not found: {workspace_dir}")

In [ ]:
# Metadata overview
run_cli([
    "metadata-overview",
    "--view", WORKSPACE,
])

In [ ]:
# Abundance matrix
run_cli([
    "abundance-matrix",
    "--view", WORKSPACE,
])

In [ ]:
# Build taxonomy lineage DB (required by alpha-diversity)
run_cli([
    "build-taxonomy-lineage",
    "--view", WORKSPACE,
])

In [ ]:
# Alpha diversity
ALPHA_RANK = "genus"
ALPHA_GROUP = None  # Example: "Group"
LINEAGE_DB_PATH = workspace_results / "_cache" / "taxonomy_lineage_db.json"

cmd = [
    "alpha-diversity",
    "--view", WORKSPACE,
    "--rank", ALPHA_RANK,
    "--lineage-db-path", str(LINEAGE_DB_PATH),
]
if ALPHA_GROUP:
    cmd += ["--group", ALPHA_GROUP]

run_cli(cmd)

In [ ]:
# Taxonomic abundance
TAXONOMIC_RANK = "genus"
TAXONOMIC_TAXON = "Prevotella"
TAXONOMIC_GROUP = None  # Example: "Group"

cmd = [
    "taxonomic-abundance",
    "--view", WORKSPACE,
    "--rank", TAXONOMIC_RANK,
    "--taxon", TAXONOMIC_TAXON,
]
if TAXONOMIC_GROUP:
    cmd += ["--group", TAXONOMIC_GROUP]

run_cli(cmd)

In [ ]:
# Single taxon abundance
SINGLE_TAXON_RANK = "superkingdom"
SINGLE_TAXON_NAME = "Viruses"
SINGLE_SOURCE_KEYS = ["sample_name", "host_subject_id_sam"]

run_cli([
    "single-taxon-abundance",
    "--view", WORKSPACE,
    "--rank", SINGLE_TAXON_RANK,
    "--taxon", SINGLE_TAXON_NAME,
    "--source-keys", *SINGLE_SOURCE_KEYS,
])